# Reconhecimento de Padrões - Trabalho 2 (Google Colab)
## Análise Comparativa de Métodos de Regressão

**Autores:**
- Alan P. Berger Saar
- Leonardo Herkenhoff

**Professor:** Dr. Sérgio Nery Simões
**Instituição:** PPCOMP - IFES

---

### Configuração do Ambiente no Google Colab

A célula abaixo detecta se o notebook está rodando no Google Colab. Se sim, instala as dependências necessárias (`ucimlrepo`, `xgboost`) e cria as pastas locais para salvar as figuras (`relatorio/img/`) e os modelos (`modelos_salvos/`).

Após a execução, você poderá baixar a pasta `relatorio/img/` com todos os gráficos gerados para o seu relatório Typst.

In [ ]:
import sys
import os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Ambiente detectado: Google Colab. Instalando dependências...")
    !pip install ucimlrepo xgboost joblib pandas numpy scikit-learn matplotlib seaborn scipy
    
    # No Colab, criamos as pastas de saída diretamente no diretório atual (/content/)
    IMG_DIR = "relatorio/img"
    MODELS_DIR = "modelos_salvos"
else:
    print("Ambiente detectado: Local.")
    IMG_DIR = "../relatorio/img"
    MODELS_DIR = "../modelos_salvos"

os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Diretório de Imagens: {IMG_DIR}")
print(f"Diretório de Modelos: {MODELS_DIR}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import joblib

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Configurações estéticas de gráficos
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 16
})

print("Bibliotecas importadas com sucesso.")

# Etapa 1 — Análise Exploratória dos Dados (EDA)

Antes de treinar qualquer modelo, precisamos entender os nossos dados. Vamos carregar o dataset diretamente do repositório da UCI e realizar análises visuais e estatísticas.

### Perguntas que a EDA responderá:
1. **Qual a distribuição da variável-alvo `cnt`?** Ela é simétrica ou possui assimetria?
2. **Por que a transformação logarítmica (`log1p`) é recomendada?**
3. **Quais características apresentam maior correlação com `cnt`?**
4. **Existe multicolinearidade?** (Por exemplo, entre `temp` e `atemp`)
5. **Como o consumo varia ao longo do dia, da semana e das estações do ano?**
6. **Quais são os impactos do clima (`weathersit`) e dias úteis (`workingday`) na demanda?**

In [ ]:
# Carregando o dataset hourly (ID 275)
print("Buscando dataset do repositório UCI...")
dataset = fetch_ucirepo(id=275)

# Removemos dteday (coluna não numérica) que não faz parte das features do enunciado
X = dataset.data.features.drop(columns=['dteday'])
y = dataset.data.targets['cnt']

# Criando dataframe único para EDA
df = X.copy()
df['cnt'] = y

print(f"Dataset carregado. Dimensões: {df.shape}")
df.info()

### 1.1 Distribuição da Variável-Alvo `cnt`

A variável-alvo `cnt` é uma variável de contagem com assimetria positiva severa.

#### A Transformação Logarítmica (`log1p`):
A transformação $y \to \log(1 + y)$ (`log1p`) estabiliza a variância (mitiga heterocedasticidade) e comprime a cauda longa de valores altos, tornando a distribuição aproximadamente normal.

In [ ]:
print(df['cnt'].describe())
print(f"Assimetria (skewness) da variável original: {df['cnt'].skew():.3f}")
print(f"Assimetria (skewness) da variável transformada log1p: {np.log1p(df['cnt']).skew():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['cnt'], bins=50, kde=True, color='steelblue', edgecolor='white', ax=axes[0])
axes[0].set_title('Distribuição de cnt (Original)')
axes[0].set_xlabel('cnt (Aluguéis de Bicicletas)')
axes[0].set_ylabel('Frequência')

sns.histplot(np.log1p(df['cnt']), bins=50, kde=True, color='darkorange', edgecolor='white', ax=axes[1])
axes[1].set_title('Distribuição de log1p(cnt)')
axes[1].set_xlabel('log1p(cnt)')
axes[1].set_ylabel('Frequência')

plt.tight_layout()
fig.savefig(f"{IMG_DIR}/distribuicao_cnt.png", dpi=150, bbox_inches="tight")
plt.show()

### 1.2 Mapa de Correlações (Heatmap)

Medimos a correlação de Pearson para identificar multicolinearidade (como entre `temp` e `atemp`).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, linewidths=0.5)
ax.set_title('Mapa de Correlações entre Variáveis')
plt.tight_layout()
fig.savefig(f"{IMG_DIR}/heatmap_correlacoes.png", dpi=150, bbox_inches="tight")
plt.show()

### 1.3 Padrões Temporais (Hora, Mês, Estação)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df.groupby('hr')['cnt'].mean().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Média de Aluguéis por Hora')
axes[0].set_xlabel('Hora do Dia')
axes[0].set_ylabel('cnt Médio')
axes[0].tick_params(axis='x', rotation=0)

df.groupby('mnth')['cnt'].mean().plot(kind='bar', ax=axes[1], color='seagreen', edgecolor='white')
axes[1].set_title('Média de Aluguéis por Mês')
axes[1].set_xlabel('Mês')
axes[1].set_ylabel('cnt Médio')
axes[1].tick_params(axis='x', rotation=0)

df.groupby('season')['cnt'].mean().plot(kind='bar', ax=axes[2], color='darkorange', edgecolor='white')
axes[2].set_title('Média de Aluguéis por Estação')
axes[2].set_xlabel('Estação')
axes[2].set_ylabel('cnt Médio')
axes[2].tick_params(axis='x', rotation=0)
axes[2].set_xticklabels(['Primavera', 'Verão', 'Outono', 'Inverno'])

plt.tight_layout()
fig.savefig(f"{IMG_DIR}/padroes_temporais.png", dpi=150, bbox_inches="tight")
plt.show()

### 1.4 Condições Climáticas e Contexto do Dia

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.boxplot(data=df, x='weathersit', y='cnt', ax=axes[0], palette='Blues')
axes[0].set_title('cnt por Condição Climática')
axes[0].set_xlabel('weathersit (1=Claro → 4=Chuva Forte)')
axes[0].set_ylabel('cnt')

sns.boxplot(data=df, x='workingday', y='cnt', ax=axes[1], palette='Set1')
axes[1].set_title('cnt — Dia Útil vs. Não Útil')
axes[1].set_xticklabels(['Fim de Semana/Feriado', 'Dia Útil'])
axes[1].set_xlabel('Tipo de Dia')
axes[1].set_ylabel('cnt')

sns.boxplot(data=df, x='weekday', y='cnt', ax=axes[2], palette='Set2')
axes[2].set_title('cnt por Dia da Semana')
axes[2].set_xlabel('Dia (0=Domingo → 6=Sábado)')
axes[2].set_ylabel('cnt')

plt.tight_layout()
fig.savefig(f"{IMG_DIR}/clima_e_contexto.png", dpi=150, bbox_inches="tight")
plt.show()

# Etapa 2 — Pré-processamento

Configuramos a divisão treino/teste e estruturamos as transformações para evitar vazamentos de dados.

In [ ]:
# Separando features e alvo
X = df.drop(columns=['cnt'])
y = df['cnt']

# Split Hold-out (80% treino, 20% teste) estratificado por 'season'
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=X['season']
)

print(f"Treino shape: {X_train.shape}, Teste shape: {X_test.shape}")

# Definindo tipos de colunas
num_cols = ['temp', 'atemp', 'hum', 'windspeed']
cat_ord_cols = ['hr', 'mnth', 'season', 'weekday', 'weathersit']
bin_cols = ['holiday', 'workingday', 'yr']

# Aplicando transformação log1p na variável alvo
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# Configurando o ColumnTransformer para pré-processamento das features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('ord', 'passthrough', cat_ord_cols),
        ('bin', 'passthrough', bin_cols)
    ]
)

print("Pré-processamento configurado via ColumnTransformer.")

# Etapa 3 — Modelos de Regressão e Tuning

Ajustaremos hiperparâmetros com `RandomizedSearchCV` e 5-fold CV no treino.

In [ ]:
best_estimators = {}
cv_tuning = KFold(n_splits=5, shuffle=True, random_state=42)
print("Configuração inicial de Tuning pronta.")

In [ ]:
# 1. OLS
ols_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])
ols_pipeline.fit(X_train, y_train_log)
best_estimators['OLS'] = ols_pipeline
print("Modelo OLS ajustado.")

In [ ]:
# 2. Ridge
ridge_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Ridge())
])
ridge_param_dist = {
    'regressor__alpha': stats.loguniform(1e-3, 1e3)
}
ridge_search = RandomizedSearchCV(
    ridge_pipeline, param_distributions=ridge_param_dist, 
    n_iter=30, cv=cv_tuning, scoring='neg_mean_squared_error', 
    random_state=42, n_jobs=-1
)
ridge_search.fit(X_train, y_train_log)
best_estimators['Ridge'] = ridge_search.best_estimator_
print(f"Ridge ajustado. Alpha: {ridge_search.best_params_['regressor__alpha']:.4f}")

In [ ]:
# 3. Lasso
lasso_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Lasso(max_iter=10000))
])
lasso_param_dist = {
    'regressor__alpha': stats.loguniform(1e-5, 1.0)
}
lasso_search = RandomizedSearchCV(
    lasso_pipeline, param_distributions=lasso_param_dist, 
    n_iter=30, cv=cv_tuning, scoring='neg_mean_squared_error', 
    random_state=42, n_jobs=-1
)
lasso_search.fit(X_train, y_train_log)
best_estimators['Lasso'] = lasso_search.best_estimator_
print(f"Lasso ajustado. Alpha: {lasso_search.best_params_['regressor__alpha']:.6f}")

In [ ]:
# 4. Random Forest
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])
rf_param_dist = {
    'regressor__n_estimators': stats.randint(50, 150),
    'regressor__max_depth': stats.randint(8, 20),
    'regressor__min_samples_split': stats.randint(2, 10),
    'regressor__max_features': ['sqrt', None]
}
rf_search = RandomizedSearchCV(
    rf_pipeline, param_distributions=rf_param_dist, 
    n_iter=30, cv=cv_tuning, scoring='neg_mean_squared_error', 
    random_state=42, n_jobs=-1
)
print("Ajustando Random Forest (isso pode levar alguns minutos)...")
rf_search.fit(X_train, y_train_log)
best_estimators['Random Forest'] = rf_search.best_estimator_
print(f"Random Forest ajustado. Melhores parâmetros: {rf_search.best_params_}")

In [ ]:
# 5. XGBoost
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(random_state=42))
])
xgb_param_dist = {
    'regressor__n_estimators': stats.randint(80, 200),
    'regressor__max_depth': stats.randint(4, 8),
    'regressor__learning_rate': stats.uniform(0.01, 0.20),
    'regressor__subsample': stats.uniform(0.6, 0.4),
    'regressor__colsample_bytree': stats.uniform(0.6, 0.4),
    'regressor__reg_alpha': stats.loguniform(1e-3, 10.0),
    'regressor__reg_lambda': stats.loguniform(1e-3, 10.0)
}
xgb_search = RandomizedSearchCV(
    xgb_pipeline, param_distributions=xgb_param_dist, 
    n_iter=30, cv=cv_tuning, scoring='neg_mean_squared_error', 
    random_state=42, n_jobs=-1
)
print("Ajustando XGBoost...")
xgb_search.fit(X_train, y_train_log)
best_estimators['XGBoost'] = xgb_search.best_estimator_
print(f"XGBoost ajustado. Melhores parâmetros: {xgb_search.best_params_}")

In [ ]:
# Salvar os modelos ajustados
for name, estimator in best_estimators.items():
    filename = os.path.join(MODELS_DIR, f"{name.lower().replace(' ', '_')}_model.pkl")
    joblib.dump(estimator, filename)
print("Modelos persistidos em:", MODELS_DIR)

# Etapa 4 — Avaliação Experimental

Fazemos a validação cruzada manual de 10 folds e a avaliação hold-out no teste com a transformação reversa `expm1()`.

In [ ]:
def calculate_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1.0))) * 100
    return rmse, mae, r2, mape

cv_results = []
holdout_results = []
cv_10fold = KFold(n_splits=10, shuffle=True, random_state=42)

for name, pipeline in best_estimators.items():
    print(f"Avaliando {name}...")
    rmse_folds, mae_folds, r2_folds, mape_folds = [], [], [], []
    
    for train_idx, val_idx in cv_10fold.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr_log, y_val = y_train_log.iloc[train_idx], y_train.iloc[val_idx]
        
        from sklearn.base import clone
        fold_pipeline = clone(pipeline)
        fold_pipeline.fit(X_tr, y_tr_log)
        y_val_pred = np.expm1(fold_pipeline.predict(X_val))
        
        rmse_f, mae_f, r2_f, mape_f = calculate_metrics(y_val, y_val_pred)
        rmse_folds.append(rmse_f)
        mae_folds.append(mae_f)
        r2_folds.append(r2_f)
        mape_folds.append(mape_f)
        
    cv_results.append({
        'Modelo': name,
        'RMSE CV': f"{np.mean(rmse_folds):.2f} ± {np.std(rmse_folds):.2f}",
        'MAE CV': f"{np.mean(mae_folds):.2f} ± {np.std(mae_folds):.2f}",
        'R2 CV': f"{np.mean(r2_folds):.3f} ± {np.std(r2_folds):.3f}",
        'MAPE CV (%)': f"{np.mean(mape_folds):.2f}% ± {np.std(mape_folds):.2f}%"
    })
    
    y_test_pred = np.expm1(pipeline.predict(X_test))
    rmse_t, mae_t, r2_t, mape_t = calculate_metrics(y_test, y_test_pred)
    holdout_results.append({
        'Modelo': name,
        'RMSE Teste': rmse_t,
        'MAE Teste': mae_t,
        'R2 Teste': r2_t,
        'MAPE Teste (%)': mape_t
    })

df_cv = pd.DataFrame(cv_results)
df_holdout = pd.DataFrame(holdout_results).sort_values(by='RMSE Teste')

print("=== Resultados 10-Fold CV ===")
print(df_cv.to_string(index=False))
print("\n=== Resultados Hold-out no Teste ===")
print(df_holdout.to_string(index=False))

# Salvar CSVs
df_cv.to_csv(os.path.join(IMG_DIR, "resultado_cv.csv"), index=False)
df_holdout.to_csv(os.path.join(IMG_DIR, "resultado_holdout.csv"), index=False)

# Etapa 5 — Diagnóstico dos Resíduos

Efetuamos a análise dos resíduos no conjunto de teste para os dois melhores modelos.

In [ ]:
top_2_names = df_holdout['Modelo'].head(2).tolist()
print(f"Melhores modelos: {top_2_names}")

In [ ]:
for model_name in top_2_names:
    pipeline = best_estimators[model_name]
    y_pred = np.expm1(pipeline.predict(X_test))
    residuals = y_test - y_pred
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f"Análise de Resíduos: {model_name}", fontsize=16)
    
    # Histograma
    sns.histplot(residuals, kde=True, bins=50, color='darkred', edgecolor='white', ax=axes[0, 0])
    axes[0, 0].set_title("Distribuição dos Resíduos")
    axes[0, 0].set_xlabel("Resíduo")
    
    # Scatter
    axes[0, 1].scatter(y_pred, residuals, alpha=0.3, color='purple', edgecolors='none')
    axes[0, 1].axhline(0, color='black', linestyle='--', linewidth=1.5)
    axes[0, 1].set_title("Resíduos vs. Valores Preditos")
    axes[0, 1].set_xlabel("Valores Preditos")
    axes[0, 1].set_ylabel("Resíduo")
    
    # Q-Q
    stats.probplot(residuals, dist="norm", plot=axes[1, 0])
    axes[1, 0].set_title("Q-Q Plot")
    
    # Temp
    axes[1, 1].scatter(X_test['temp'], residuals, alpha=0.3, color='teal', edgecolors='none')
    axes[1, 1].axhline(0, color='black', linestyle='--', linewidth=1.5)
    axes[1, 1].set_title("Resíduos vs. Temperatura")
    axes[1, 1].set_xlabel("Temperatura")
    axes[1, 1].set_ylabel("Resíduo")
    
    plt.tight_layout()
    m_id = model_name.lower().replace(' ', '_')
    fig.savefig(os.path.join(IMG_DIR, f"residuos_{m_id}.png"), dpi=150, bbox_inches="tight")
    plt.show()
    
    # Teste de Normalidade
    stat, p_val = stats.normaltest(residuals)
    print(f"\n=== Teste de Normalidade para {model_name} ===")
    print(f"Estatística: {stat:.4f}, p-valor: {p_val:.4e}")

In [ ]:
for model_name in top_2_names:
    pipeline = best_estimators[model_name]
    y_pred = np.expm1(pipeline.predict(X_test))
    residuals = y_test - y_pred
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Resíduos vs. Features Temporais: {model_name}", fontsize=15)
    
    sns.boxplot(x=X_test['hr'], y=residuals, ax=axes[0], color='lightblue')
    axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)
    axes[0].set_title("Resíduos por Hora do Dia")
    axes[0].set_xlabel("Hora")
    axes[0].set_ylabel("Resíduo")
    
    sns.boxplot(x=X_test['season'], y=residuals, ax=axes[1], color='lightgreen')
    axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
    axes[1].set_title("Resíduos por Estação")
    axes[1].set_xlabel("Estação")
    axes[1].set_ylabel("Resíduo")
    axes[1].set_xticklabels(['Primavera', 'Verão', 'Outono', 'Inverno'])
    
    plt.tight_layout()
    m_id = model_name.lower().replace(' ', '_')
    fig.savefig(os.path.join(IMG_DIR, f"residuos_temporal_{m_id}.png"), dpi=150, bbox_inches="tight")
    plt.show()